# Notebook 3: HAR-RV Model
**Project:** GARCH & Volatility Forecasting
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from arch import arch_model
import statsmodels.api as sm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
spy = returns['SPY'] * 100
spy_raw = returns['SPY']
print(f'Loaded {len(spy):,} obs | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 obs | 2015-01-01 to 2024-12-31


In [2]:
from har_rv import compute_rv_components, fit_har_rv
comps = compute_rv_components(spy_raw)
har_r = fit_har_rv(spy_raw)
print(f'HAR-RV R-squared: {har_r["r_squared"]}')
print('Parameters:')
print(har_r['params'].round(6))
har_r['params'].to_frame('value').to_csv('../results/har_rv_parameters.csv')

HAR-RV R-squared: 0.0766
Parameters:
const    0.000053
RV      -0.065804
RV_w     0.445964
RV_m     0.179409
dtype: float64


In [3]:
fitted=har_r['fitted']; y=comps['RV'].shift(-1).dropna()
fig,axes=plt.subplots(2,1,figsize=(13,9))
ax1=axes[0]
ax1.plot(y.index,np.sqrt(y)*np.sqrt(252)*100,color='#888',linewidth=0.8,alpha=0.7,label='Realised Vol (%)')
ax1.plot(fitted.index,np.sqrt(np.abs(fitted))*np.sqrt(252)*100,color='#E24B4A',linewidth=1.3,label='HAR-RV Fitted')
ax1.set_title(f'HAR-RV Fitted vs Realised Volatility | R²={har_r["r_squared"]}'); ax1.legend(fontsize=9)
ax2=axes[1]
p=har_r['params']
ax2.bar(['Daily','Weekly','Monthly'],[p['RV'],p['RV_w'],p['RV_m']],
        color=['#185FA5','#E24B4A','#1D9E75'],alpha=0.85,width=0.5)
ax2.set_title('HAR-RV Coefficients — Volatility Memory Across Time Horizons')
plt.tight_layout(); plt.savefig('../results/04_har_rv_fit.png',dpi=150,bbox_inches='tight')
plt.show()